# Churn Prediction Pipeline - Production Ready
Minimal orchestration notebook using modular src components

In [ ]:
import sys
sys.path.insert(0, '.')
from src.utils import print_section
from src.data_loader import load_data, prepare_features, split_data
from src.preprocessing import preprocess_pipeline
from src.models import train_grid_search, train_bayesian_search, save_models, load_models
from src.evaluation import compare_models, plot_confusion_matrices, plot_roc_curves
from src.explainability import shap_analysis_pipeline

In [ ]:
# RUN COMPLETE PIPELINE
print_section('DATA LOADING & PREPROCESSING')
df = load_data()
X, y = prepare_features(df)
X_train, X_test, y_train, y_test = split_data(X, y)
X_train_scaled, X_test_scaled, y_train_sm, y_test, scaler = preprocess_pipeline(X_train, X_test, y_train, y_test)

print_section('MODEL TRAINING')
grid_search, grid_time = train_grid_search(X_train_scaled, y_train_sm)
bayes_search, bayes_time = train_bayesian_search(X_train_scaled, y_train_sm)

print_section('MODEL EVALUATION')
compare_models(grid_search.best_estimator_, bayes_search.best_estimator_, X_test_scaled, y_test)

print_section('SAVING MODELS')
save_models(grid_search, bayes_search, scaler)

In [ ]:
# VISUALIZATIONS & EXPLAINABILITY
print_section('VISUALIZATIONS')
plot_confusion_matrices(grid_search.best_estimator_, bayes_search.best_estimator_, X_test_scaled, y_test)
plot_roc_curves(grid_search.best_estimator_, bayes_search.best_estimator_, X_test_scaled, y_test)

print_section('SHAP FEATURE IMPORTANCE')
shap_analysis_pipeline(grid_search.best_estimator_, bayes_search.best_estimator_, X_test_scaled, X.columns)

In [ ]:
# LOAD MODELS FOR INFERENCE
print_section('INFERENCE EXAMPLE')
grid_model, bayes_model, loaded_scaler = load_models()
print('✓ Models loaded successfully')
print(f'  Grid Model: {grid_model}')
print(f'  Bayesian Model: {bayes_model}')